# Поиск статей справки Авито — Описание решения

**Финальная метрика MAP@10 на test.f:** 0.480  
**Стек:** Python, PyTorch, Pandas, BeautifulSoup4, emoji, nltk, pymorphy3, rank_bm25, sentence-transformers.

## 1. Загрузка данных и первичная обработка

In [14]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

tqdm.pandas()

# Загрузка данных
articles = pd.read_feather('articles.f')
calibration = pd.read_feather('calibration.f')
test = pd.read_feather('test.f')

print(f"Размер корпуса статей: {len(articles)}")
print(f"Размер калибровочной выборки: {len(calibration)}")
print(f"Размер тестовой выборки: {len(test)}")

Размер корпуса статей: 793
Размер калибровочной выборки: 500
Размер тестовой выборки: 500


In [15]:
def clean_html(html_text):
    """
    Очистка HTML от тегов, скрытых скриптов и стилей.
    """
    if pd.isna(html_text):
        return ""
    soup = BeautifulSoup(html_text, "html.parser")
    
    # Удаление системного мусора, не несущего смысловой нагрузки
    for script_or_style in soup(['script', 'style']):
        script_or_style.decompose()
        
    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Очистка HTML корпуса статей...")
articles['clean_body'] = articles['body'].progress_apply(clean_html)

# Title Boosting: Искусственно увеличиваем вес заголовка для лексического поиска
# Дублирование заголовка 3 раза позволяет BM25 фокусироваться на главной теме статьи
articles['full_text'] = articles['title'] + ". " + articles['title'] + ". " + articles['title'] + ". " + articles['clean_body']

Очистка HTML корпуса статей...


  0%|          | 0/793 [00:00<?, ?it/s]

## 2. Локальная метрика (MAP@10)
Для объективной оценки всех гипотез реализован локальный подсчет целевой метрики.

In [17]:
def apk(actual, predicted, k=10):
    if not actual:
        return 0.0
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(actual), k)

def mapk(actual_list, predicted_list, k=10):
    return np.mean([apk(a, p, k) for a, p in zip(actual_list, predicted_list)])

# Преобразование ground_truth в списки integer для быстрого подсчета
calibration['ground_truth_list'] = calibration['ground_truth'].apply(lambda x: [int(i) for i in str(x).split()])

## 3. Этап 1: Лексический поиск (BM25)
Для устранения проблемы флективности русского языка применена лемматизация. Также подобраны гиперпараметры `BM25Okapi` для снижения штрафа за длину гигантских статей.

In [18]:
import nltk
from nltk.corpus import stopwords
import pymorphy3
from rank_bm25 import BM25Okapi

# Инициализация лемматизатора и стоп-слов
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('russian'))
morph = pymorphy3.MorphAnalyzer()

def advanced_tokenize(text):
    """
    Токенизация с приведением к нижнему регистру, удалением пунктуации, 
    стоп-слов и лемматизацией (pymorphy3).
    """
    if pd.isna(text):
        return []
    
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = text.split()
    
    lemmatized_tokens = []
    for word in tokens:
        if word not in stop_words:
            normal_form = morph.parse(word)[0].normal_form
            lemmatized_tokens.append(normal_form)
            
    return lemmatized_tokens

print("Лемматизация корпуса статей...")
corpus_tokens = articles['full_text'].progress_apply(advanced_tokenize).tolist()

print("Построение индекса BM25...")
# Тюнинг параметров: b=0.5 снижает штраф за объемные статьи
bm25 = BM25Okapi(corpus_tokens, k1=2.0, b=0.5)
article_ids = articles['article_id'].tolist()

def get_bm25_top_k(query, k=10):
    query_tokens = advanced_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:k]
    return [article_ids[i] for i in top_indices]

# Локальная валидация бейзлайна
bm25_preds = calibration['query_text'].progress_apply(lambda q: get_bm25_top_k(q, k=10))
print(f"Baseline MAP@10 (BM25): {mapk(calibration['ground_truth_list'].tolist(), bm25_preds.tolist(), k=10):.4f}")

Лемматизация корпуса статей...


  0%|          | 0/793 [00:00<?, ?it/s]

Построение индекса BM25...


  0%|          | 0/500 [00:00<?, ?it/s]

Baseline MAP@10 (BM25): 0.3065


## 4. Этап 2: Семантический поиск (Dense Retrieval + Max Pooling)
Для решения проблемы "размытия" эмбеддингов на длинных текстах реализована стратегия Chunking + Max Pooling. Использована мультиязычная модель `BAAI/bge-m3`. Инференс проводится в формате `fp16` для оптимизации VRAM.

In [20]:
import torch
import gc
from sentence_transformers import SentenceTransformer, util

torch.cuda.empty_cache()
gc.collect()

print("Загрузка Bi-Encoder модели BAAI/bge-m3 (fp16)...")
embedder = SentenceTransformer('BAAI/bge-m3', device='cuda', model_kwargs={"torch_dtype": torch.float16})
embedder.max_seq_length = 2048

def get_chunks(text, chunk_size=200, overlap=50):
    """Нарезка текста на чанки с перекрытием для сохранения контекста на стыках"""
    words = text.split()
    if not words:
        return [""]
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i + chunk_size]))
        if i + chunk_size >= len(words):
            break
    return chunks

print("Нарезка корпуса на чанки...")
chunk_records = []
for _, row in articles.iterrows():
    title = row['title']
    # Чанкинг применяется к чистому тексту (без искусственных дублей заголовка)
    chunks = get_chunks(row['clean_body'], chunk_size=200, overlap=50)
    
    for chunk in chunks:
        chunk_records.append({
            'article_id': row['article_id'],
            'chunk_text': f"{title}. {chunk}"
        })
        
chunk_df = pd.DataFrame(chunk_records)
print(f"Сформировано {len(chunk_df)} чанков.")

print("Векторизация чанков на GPU...")
chunk_embeddings = embedder.encode(
    chunk_df['chunk_text'].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)
chunk_article_ids = chunk_df['article_id'].values

def get_vector_top_k(query, k=50):
    query_embedding = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    cos_scores = util.cos_sim(query_embedding, chunk_embeddings)[0].cpu().numpy()
    
    # Max Pooling: итоговый скор статьи равен скору её наиболее релевантного чанка
    article_max_scores = {}
    for idx, score in enumerate(cos_scores):
        doc_id = chunk_article_ids[idx]
        if doc_id not in article_max_scores or score > article_max_scores[doc_id]:
            article_max_scores[doc_id] = score
            
    sorted_docs = sorted(article_max_scores.keys(), key=lambda x: article_max_scores[x], reverse=True)
    return sorted_docs[:k]

# Валидация семантического поиска
vec_preds = calibration['query_text'].progress_apply(lambda q: get_vector_top_k(q, k=10))
print(f"Semantic Search MAP@10: {mapk(calibration['ground_truth_list'].tolist(), vec_preds.tolist(), k=10):.4f}")

Загрузка Bi-Encoder модели BAAI/bge-m3 (fp16)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Нарезка корпуса на чанки...
Сформировано 4083 чанков.
Векторизация чанков на GPU...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

Semantic Search MAP@10: 0.4475


## 5. Гибридный поиск (Reciprocal Rank Fusion)
Поскольку лексический и семантический поиск оперируют в разных шкалах (BM25 vs Cosine Similarity), их результаты объединяются с помощью алгоритма взвешенного RRF (веса подобраны эмпирически: 1.0 для BM25 и 3.1 для векторов).

In [21]:
def get_hybrid_rrf_top_k(query, k=10, rrf_k=60):
    # Кандидаты от лексического поиска
    query_tokens = advanced_tokenize(query)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:50]
    bm25_candidates = [article_ids[i] for i in bm25_top_indices]
    
    # Кандидаты от семантического поиска
    vector_candidates = get_vector_top_k(query, k=50)
    
    # Слияние через RRF
    rrf_scores = {}
    
    for rank, doc_id in enumerate(bm25_candidates):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)
        
    for rank, doc_id in enumerate(vector_candidates):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 3.1 / (rrf_k + rank + 1)
        
    sorted_docs = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return sorted_docs[:k]

# Валидация гибридного поиска
hybrid_preds = calibration['query_text'].progress_apply(lambda q: get_hybrid_rrf_top_k(q, k=10))
print(f"Hybrid Search MAP@10 (BM25 + Vectors + RRF): {mapk(calibration['ground_truth_list'].tolist(), hybrid_preds.tolist(), k=10):.4f}")

  0%|          | 0/500 [00:00<?, ?it/s]

Hybrid Search MAP@10 (BM25 + Vectors + RRF): 0.4833


## 6. Финальное ранжирование (Cross-Encoder)
Для максимизации точности применяется подход Retrieve & Rerank. В качестве реранкера используется `BAAI/bge-reranker-v2-m3`. Для 100% утилизации GPU реализован батчевый процессинг формирования пар `[Query, Document]`.

In [22]:
from sentence_transformers import CrossEncoder

torch.cuda.empty_cache()
gc.collect()

print("Загрузка Cross-Encoder модели BAAI/bge-reranker-v2-m3 (fp16)...")
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=1024, device='cuda')
reranker.model.half()

# Формирование словаря чистых текстов (без искусственных дублей заголовка) для реранкера
article_text_dict = dict(zip(articles['article_id'], articles['title'] + " " + articles['clean_body']))

print("Сбор кандидатов (Топ-30) от гибридной системы для калибровочного датасета...")
all_pairs = []
pair_metadata = []
queries = calibration['query_text'].tolist()

for q_idx, query in enumerate(tqdm(queries)):
    candidates = get_hybrid_rrf_top_k(query, k=30, rrf_k=60)
    for doc_id in candidates:
        all_pairs.append([query, article_text_dict[doc_id]])
        pair_metadata.append((q_idx, doc_id))

print(f"Сформировано {len(all_pairs)} пар. Запуск батчевого инференса Cross-Encoder...")
all_scores = reranker.predict(all_pairs, batch_size=256, show_progress_bar=True)

print("Сборка финального ранжирования...")
query_results = {i: [] for i in range(len(queries))}
for (q_idx, doc_id), score in zip(pair_metadata, all_scores):
    query_results[q_idx].append((doc_id, score))

reranked_predictions = []
for q_idx in range(len(queries)):
    sorted_docs = sorted(query_results[q_idx], key=lambda x: x[1], reverse=True)
    top_10_ids = [doc_id for doc_id, score in sorted_docs[:10]]
    reranked_predictions.append(top_10_ids)

reranked_map10 = mapk(calibration['ground_truth_list'].tolist(), reranked_predictions, k=10)
print(f"Финальный MAP@10 (Hybrid + Reranker) на калибровке: {reranked_map10:.4f}")

Загрузка Cross-Encoder модели BAAI/bge-reranker-v2-m3 (fp16)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Сбор кандидатов (Топ-30) от гибридной системы для калибровочного датасета...


  0%|          | 0/500 [00:00<?, ?it/s]

Сформировано 15000 пар. Запуск батчевого инференса Cross-Encoder...


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Сборка финального ранжирования...
Финальный MAP@10 (Hybrid + Reranker) на калибровке: 0.4760


## 7. Инференс на тестовой выборке
Применение настроенного пайплайна к `test.f` и генерация сабмита.

In [23]:
torch.cuda.empty_cache()
gc.collect()

print("Сбор кандидатов (Топ-25) от гибридной системы для TEST датасета...")
all_test_pairs = []
test_pair_metadata = [] 
test_queries = test['query_text'].tolist()

for q_idx, query in enumerate(tqdm(test_queries)):
    candidates = get_hybrid_rrf_top_k(query, k=25, rrf_k=60)
    for doc_id in candidates:
        all_test_pairs.append([query, article_text_dict[doc_id]])
        test_pair_metadata.append((q_idx, doc_id))

print(f"Сформировано {len(all_test_pairs)} пар. Запуск Cross-Encoder...")
test_all_scores = reranker.predict(all_test_pairs, batch_size=256, show_progress_bar=True)

test_query_results = {i: [] for i in range(len(test_queries))}
for (q_idx, doc_id), score in zip(test_pair_metadata, test_all_scores):
    test_query_results[q_idx].append((doc_id, score))

final_answers = []
for q_idx in range(len(test_queries)):
    sorted_docs = sorted(test_query_results[q_idx], key=lambda x: x[1], reverse=True)
    top_10_ids = [str(doc_id) for doc_id, score in sorted_docs[:10]]
    final_answers.append(" ".join(top_10_ids))

# Сохранение результатов
test['answer'] = final_answers
test[['query_id', 'answer']].to_csv('answer_reranked.csv', index=False)
print("Готово. Файл answer_reranked.csv успешно сгенерирован.")

Сбор кандидатов (Топ-25) от гибридной системы для TEST датасета...


  0%|          | 0/500 [00:00<?, ?it/s]

Сформировано 12500 пар. Запуск Cross-Encoder...


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Готово. Файл answer_reranked.csv успешно сгенерирован.


## 8. Эволюция пайплайна и инженерные оптимизации

В процессе разработки решение прошло через несколько итераций, направленных на повышение метрики и оптимизацию производительности под аппаратные ограничения (NVIDIA RTX 5060 Ti 16GB).

### Итеративное улучшение качества:
1. **EDA и очистка данных:** Изначальный прогон лексического поиска показал метрику ~0.30. Визуальный анализ корпуса выявил наличие скрытых тегов (`script`, `style`) и небуквенных символов (эмодзи), которые выступали шумом для `bge-m3`. Тотальная очистка через `BeautifulSoup` и `emoji` дала первый прирост качества.
2. **Тюнинг RRF:** Изначальное слияние лексики и семантики с равными весами не давало максимального эффекта. Анализ показал, что семантический поиск справляется с разрывом словаря лучше, поэтому веса в алгоритме RRF были смещены (1.0 для BM25 и 3.1 для векторов).
3. **Борьба с "Lost in the Middle":** Увеличение окна реранкера до максимума приводило к падению метрики, так как модель "слепла" на мусорных данных в подвалах огромных статей. Оптимальным решением стало удержание `max_length` реранкера на уровне 512-1024 токенов, что заставляет модель фокусироваться на заголовках и первых абзацах.

### Хардверная оптимизация (CPU Bottleneck & VRAM OOM):
Изначальная архитектура инференса строилась через последовательный `progress_apply`. Это привело к двум критическим проблемам:
* **Квадратичная сложность Attention:** При больших значениях `max_length` память GPU (16 ГБ) мгновенно переполнялась, и происходила выгрузка тензоров в Shared RAM оперативной памяти, что замедляло инференс до 1 часа.
* **CPU Bottleneck:** Процессор подготавливал данные по одному запросу, из-за чего GPU простаивал (утилизация падала до 8-10%).

**Решение:** 
Инференс был переписан на пакетную (batch) обработку. Все 25 000 пар `[Query, Document]` собираются в единый пул заранее. Перевод весов реранкера в `fp16` (`torch.float16`) и подбор `batch_size=256` позволили утилизировать 15.5 из 16 ГБ видеопамяти и держать загрузку GPU на уровне 100%. Скорость обработки 500 запросов реранкером сократилась **с 1 часа до ~10 минут**.

## 9. Анализ ошибок и Future Work

Текущий пайплайн (BM25 + BGE-M3 + RRF + BGE-Reranker) показал высокую эффективность (MAP@10 ~0.48 на скрытом тесте), однако имеет потенциал для развития. Основная сложность заключается в использовании Zero-Shot моделей, которые не адаптированы к доменной специфике Авито (терминология ПВЗ, логистики, статусов блокировок).

**Архитектурные шаги для пробития потолка метрики:**

1. **Доменная адаптация (Fine-Tuning) эмбеддера:**
   Так как 500 примеров из калибровки недостаточно для обучения моделей масштаба 567M параметров (риск переобучения), я бы применил генерацию синтетических Q&A пар через LLM на основе статей базы. Далее — использование BM25 для Hard Negatives Mining и дообучение `BGE-M3` с `MultipleNegativesRankingLoss`.

2. **Learning-to-Rank (LTR) через Градиентный Бустинг:**
   Вместо тяжелого Cross-Encoder (или поверх него) можно внедрить классический ML-подход. Используя `CatBoost` или `XGBoost` с функцией потерь `YetiRank`, можно обучить легковесную модель ранжирования. В качестве фичей (признаков) выступят: скоры BM25, косинусные расстояния `bge-m3`, длина статьи, BM25 по заголовкам, TF-IDF совпадения. Бустинговые деревья работают с табличными фичами в сотни раз быстрее тяжелых нейросетей, что критически важно для высоконагруженного продакшена Авито.